In [ ]:
from tenerife_rag.config import LLMConfig

config = LLMConfig.from_env()
client = config.client()

config.public_dict()

response = client.responses.create(model=config.generation_model, input="Hello, world!", temperature=config.temperature, top_p=config.top_p)
print(response.output_text)

Hello! How can I assist you today?


In [12]:
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pages = PyMuPDFLoader("../data/TENERIFE.pdf").load()
splitter = RecursiveCharacterTextSplitter(chunk_size=800, chunk_overlap=150)
windows = splitter.split_documents(pages)

print(len(windows))
print(windows[0].page_content)
print("-----------------------------------------------")
print(windows[32].page_content, windows[32].metadata)


35
TENERIFE – LUGARES DE INTERÉS 
SITIOS QUE VER 
 
ZONA NORTE 
 
• Santa Cruz de Tenerife: 
Santa Cruz de Tenerife es la capital de la isla. Quizás la ruta a seguir si vais a Santa 
Cruz sería: 
- 
Aparcar en el aparcamiento del Parque Marítimo (ubicación). 
- 
Caminar por la Avenida Marítima hasta Plaza de España (ubicación). 
- 
Por el camino de la Avenida Marítima, ver el auditorio de Tenerife (ubicación). 
- 
Una vez llegados a Plaza España, callejear un poco (subir la Calle Castillo 
dirección Plaza Weyler - ubicación –; ir hacia el Parque García Sanabria - 
ubicación -; y bajar de nuevo hacia Plaza de España pasando por la Plaza del 
Príncipe - ubicación). 
- 
Volver de nuevo a por el coche. 
- 
Ir hacia la playa de Las Teresitas (ubicación).
-----------------------------------------------
Nota: los Guachinches son típicos en la zona norte de Tenerife. 
Actualmente existen muchos restaurantes que se llaman Guachinches 
pero no son los originales, ya que tienen una carta mucho má

In [16]:
from langchain_community.vectorstores import InMemoryVectorStore

store = InMemoryVectorStore(config.embeddings())
store.add_documents(windows)

def search(query: str, top_k: int=3):
    return store.similarity_search_with_score(query, k=top_k)

In [17]:
search("playa de arena negra")

[(Document(id='cd5935dc-17b4-4329-9460-c0cb4dd20836', metadata={'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'source': '../data/TENERIFE.pdf', 'file_path': '../data/TENERIFE.pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Microsoft Word - TENERIFE.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-07-13T20:00:01+00:00', 'trapped': '', 'modDate': "D:20250713200001+00'00'", 'creationDate': "D:20250713200001+00'00'", 'page': 7}, page_content='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en la isla. \nPara llegar a esta playa, aparcar por aquí y, para acceder a la playa, se accede \ndesde aquí (puede ser que os encontréis con el acceso cortado con una valla, \npero da igual, podéis acceder igualmente). \n• La Orotava \nEste es el pueblo más bonito y maravilloso del mundo (no es porque yo sea de allí).'),
  0.581292018081072

In [19]:
search("hay carta en un guachinche")

[(Document(id='15a05533-38b7-4d0f-8e16-2498398be4ec', metadata={'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'source': '../data/TENERIFE.pdf', 'file_path': '../data/TENERIFE.pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Microsoft Word - TENERIFE.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-07-13T20:00:01+00:00', 'trapped': '', 'modDate': "D:20250713200001+00'00'", 'creationDate': "D:20250713200001+00'00'", 'page': 23}, page_content='atravesaréis una especie de camino de tierra, meteos sin miedo. Si \nvais a Tenerife y no probáis una experiencia en Guachinche, no habéis \nido a Tenerife. Os podría recomendar toda la carta, prácticamente – \nde hecho, no. No podría, ya que en un guachinche no hay carta. Y de \nbebidas solo hay tres cosas: vino, seven up para mezclar con el vino, y \nagua. That’s all. Mi recomendación realmente es que preguntéis a los \ncamareros y, si es vuestra primera experiencia en un guach

In [31]:
INSTRUCTIONS: str = """Eres un guía turístico de Tenerife. Responde en español usando SOLO los fragmentos.
Cita cada dato con el id del fragmento entre corchetes, por ejemplo [S1].
Si los fragmentos no contienen la respuesta, dilo."""

def answer(question:str, top_k: int=3):
    hits = search(question, top_k)
    context = "\n".join(
        f"[S{i}] (p. {doc.metadata['page'] + 1}) {doc.page_content}" for i, (doc, _) in enumerate(hits, 1)
    )
    
    response = client.responses.create(
        model=config.generation_model,
        instructions=INSTRUCTIONS,
        input=f"Fragmentos:\n{context}\n\nPregunta: {question}\n\nRespuesta:",
        temperature=config.temperature,
        top_p=config.top_p,
        max_output_tokens=config.max_output_tokens
    )
    
    return response.output_text, hits

In [ ]:
answer("¿Qué playas de arena negra hay en Tenerife?", top_k=3)

('La playa de arena negra que te recomiendo es la Playa de Benijo. Es conocida por tener uno de los atardeceres más espectaculares de la isla, especialmente si el día está despejado. Para llegar, puedes aparcar en un punto cercano y acceder a la playa aunque el acceso esté cortado con una valla, ya que se puede pasar igualmente [S1].',
 [(Document(id='cd5935dc-17b4-4329-9460-c0cb4dd20836', metadata={'producer': '', 'creator': 'Microsoft Word', 'creationdate': '2025-07-13T20:00:01+00:00', 'source': '../data/TENERIFE.pdf', 'file_path': '../data/TENERIFE.pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Microsoft Word - TENERIFE.docx', 'author': '', 'subject': '', 'keywords': '', 'moddate': '2025-07-13T20:00:01+00:00', 'trapped': '', 'modDate': "D:20250713200001+00'00'", 'creationDate': "D:20250713200001+00'00'", 'page': 7}, page_content='o Playa de Benijo [vídeo - ubicación] \nPlaya de arena negra, si el día está despejado, de los atardeceres más \nespectaculares que podéis ver en 